# Homework 5 - Lab Assignment
**Mateo Larrea - Music 320 - Fall 2025**

**Topic:** A Simple Guitar String Simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Audio
%matplotlib inline

## Part I: Analysis

### Question 1: Transfer Function H(z)

Given the difference equation:
$$y[n] = x[n] + ay[n - M]$$

Taking the z-transform:
$$Y(z) = X(z) + aY(z)z^{-M}$$

Solving for the transfer function:
$$H(z) = \frac{Y(z)}{X(z)} = \frac{1}{1 - az^{-M}}$$

Or equivalently:
$$H(z) = \frac{z^M}{z^M - a}$$

### Question 2: Poles of H(z)

The poles occur when the denominator equals zero:
$$z^M - a = 0$$
$$z^M = a$$

The M poles are:
$$p_i = a^{1/M} e^{j2\pi i/M} \quad \text{for } i = 0, 1, ..., M-1$$

**a) In polar coordinates:** $p_i = r_i e^{j\theta_i}$ where:
- $r_i = a^{1/M}$ (all poles have the same radius)
- $\theta_i = \frac{2\pi i}{M}$ (poles are evenly spaced around a circle)

**The smallest non-zero angle is:** $\theta_1 = \frac{2\pi}{M}$

### Question 2b: Plot poles for M=10, a=0.9^10

In [ ]:
M = 10
a = 0.9**10

r = a**(1/M)
theta = 2 * np.pi * np.arange(M) / M
poles = r * np.exp(1j * theta)

plt.figure(figsize=(8, 8))
plt.plot(poles.real, poles.imag, 'rx', markersize=12, markeredgewidth=2, label='Poles')

theta_circle = np.linspace(0, 2*np.pi, 1000)
plt.plot(np.cos(theta_circle), np.sin(theta_circle), 'b--', linewidth=1, alpha=0.5, label='Unit Circle')

plt.grid(True, alpha=0.3)
plt.axis('equal')
plt.xlabel('Real Part')
plt.ylabel('Imaginary Part')
plt.title(f'Pole-Zero Plot for M={M}, a={a:.4f}')
plt.legend()
plt.axhline(y=0, color='k', linewidth=0.5)
plt.axvline(x=0, color='k', linewidth=0.5)
plt.show()

print(f"Pole radius r = a^(1/M) = {r:.4f}")
print(f"Smallest non-zero angle θ₁ = 2π/M = {theta[1]:.4f} radians = {np.degrees(theta[1]):.1f} degrees")

### Question 3: Amplitude Response

Compute and plot the amplitude response for M=10, a=0.9^10.

The frequency response is obtained by evaluating H(z) on the unit circle: $z = e^{j\omega}$
$$H(e^{j\omega}) = \frac{1}{1 - ae^{-j\omega M}}$$

In [ ]:
omega = np.linspace(0, np.pi, 1000)
H = 1 / (1 - a * np.exp(-1j * omega * M))
amplitude_response = np.abs(H)

plt.figure(figsize=(12, 5))
plt.plot(omega / np.pi, amplitude_response, 'b-', linewidth=2)
plt.xlabel('Normalized Frequency (×π rad/sample)')
plt.ylabel('Magnitude')
plt.title(f'Amplitude Response for M={M}, a={a:.4f}')
plt.grid(True, alpha=0.3)
plt.xlim([0, 1])
plt.show()

first_peak_idx = np.argmax(amplitude_response[:len(amplitude_response)//M + 50])
first_peak_omega = omega[first_peak_idx]
print(f"First peak location: ω = {first_peak_omega:.4f} rad/sample = {first_peak_omega/np.pi:.4f}π")
print(f"r_i = {r:.4f} controls peak sharpness (closer to 1 → sharper peaks)")
print(f"θ_i controls peak spacing (θ₁ = 2π/M sets fundamental frequency)")

### Question 4: Normalized Radian Frequency Analysis

**a) Normalized radian frequency of sinusoid with period M:**

If the period is M samples, then the frequency is:
$$\omega = \frac{2\pi}{M} \text{ rad/sample}$$

**b) Frequency in Hz when sampling frequency is f_s:**
$$f = \frac{f_s}{M} \text{ Hz}$$

**c) Relation to first peak:**

This frequency corresponds exactly to the first peak of the amplitude response! The first pole is at $\theta_1 = 2\pi/M$, which creates a peak in the frequency response at $\omega = 2\pi/M$.

### Question 5: Finding M for A4 (440 Hz) and B4 (493.88 Hz)

Given $f_s = 8$ kHz, we need the first peak at frequency $f$:
$$f = \frac{f_s}{M} \Rightarrow M = \frac{f_s}{f}$$

In [ ]:
fs = 8000

f_A4 = 440
M_A4 = int(np.round(fs / f_A4))
f_A4_actual = fs / M_A4

f_B4 = 440 * 2**(2/12)
M_B4 = int(np.round(fs / f_B4))
f_B4_actual = fs / M_B4

print(f"A4: M = {M_A4}, f = {f_A4_actual:.2f} Hz (error: {f_A4_actual - f_A4:.2f} Hz)")
print(f"B4: M = {M_B4}, f = {f_B4_actual:.2f} Hz (error: {f_B4_actual - f_B4:.2f} Hz)")

### Question 6: Finding κ for Amplitude Decay

We want the amplitude to decay by a factor of 10 over one second.

Given $a = \kappa^M$ and the signal passes through the delay M times per second (at $f_s = 8$ kHz):
- Number of times through feedback loop in 1 second = $f_s / M$ times
- After 1 second, amplitude is multiplied by $a^{f_s/M} = (\kappa^M)^{f_s/M} = \kappa^{f_s}$

We want:
$$\kappa^{f_s} = \frac{1}{10}$$
$$\kappa = 10^{-1/f_s}$$

In [ ]:
kappa = 10**(-1/fs)
print(f"κ = {kappa:.10f}")
print(f"Verification: κ^{fs} = {kappa**fs:.4f} ≈ 0.1")

## Part II: Simulation

### Question 7: Guitar Function Implementation

Implement a function that simulates a plucked guitar string using the damped feedback comb filter.

In [ ]:
def Guitar(N, M, kappa):
    x = np.random.randn(M)
    a = kappa**M
    y = np.zeros(N)
    
    for n in range(N):
        if n < M:
            y[n] = x[n]
        else:
            y[n] = a * y[n - M]
    
    return y

### Question 8: Plot Guitar Output

Plot the output for N=16,000 samples using M for A4 and κ from above.

In [ ]:
N = 16000
y_guitar = Guitar(N, M_A4, kappa)
t = np.arange(N) / fs

plt.figure(figsize=(14, 5))
plt.plot(t, y_guitar, 'b-', linewidth=0.5)
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude')
plt.title(f'Simulated Guitar String: A4 ({f_A4_actual:.1f} Hz), M={M_A4}, κ={kappa:.6f}')
plt.grid(True, alpha=0.3)
plt.xlim([0, t[-1]])
plt.show()

plt.figure(figsize=(14, 5))
t_zoom = 0.1
n_zoom = int(t_zoom * fs)
plt.plot(t[:n_zoom], y_guitar[:n_zoom], 'b-', linewidth=1)
plt.xlabel('Time (seconds)')
plt.ylabel('Amplitude')
plt.title(f'Simulated Guitar String (First {t_zoom} seconds)')
plt.grid(True, alpha=0.3)
plt.show()

display(Audio(y_guitar, rate=fs))

## Part III: Making Music

### Question 9: Frequency Function

Write a function that converts note names to frequencies.

In [ ]:
def freq(note):
    semitones = {'C': -9, 'D': -7, 'E': -5, 'F': -4, 'G': -2, 'A': 0, 'B': 2}
    return 440 * 2**(semitones[note] / 12)

In [ ]:
for note in ['C', 'D', 'E', 'F', 'G', 'A', 'B']:
    print(f"{note}4: {freq(note):.2f} Hz")

### Question 10: PlayNotes Function

Write a function that plays a sequence of notes.

In [ ]:
def PlayNotes(notes, fs, octave_shift=0):
    output = np.zeros(fs * (len(notes) + 1))
    kappa_local = 10**(-1/fs)
    octave_mult = 2**octave_shift
    
    for i, note in enumerate(notes):
        if note == 'P':
            continue
        
        f_note = freq(note) * octave_mult
        M_note = int(np.round(fs / f_note))
        note_signal = Guitar(2 * fs, M_note, kappa_local)
        
        start_idx = i * fs
        end_idx = start_idx + len(note_signal)
        output[start_idx:end_idx] += note_signal
    
    max_val = np.max(np.abs(output))
    if max_val > 0:
        output = output / max_val * 0.9
    
    return Audio(output, rate=fs)

### Question 11: Test with Song

Play "Twinkle Twinkle Little Star" in octave 3.

In [ ]:
song = ['C', 'C', 'G', 'G', 'A', 'A', 'G', 'P',
        'F', 'F', 'E', 'E', 'D', 'D', 'C', 'P',
        'G', 'G', 'F', 'F', 'E', 'E', 'D', 'P',
        'G', 'G', 'F', 'F', 'E', 'E', 'D', 'P',
        'C', 'C', 'G', 'G', 'A', 'A', 'G', 'P',
        'F', 'F', 'E', 'E', 'D', 'D', 'C']

display(PlayNotes(song, fs, octave_shift=-1))